# Phase 1b (Kaggle): second training seed for the pilot comparison

**Why:** the first pilot run found Qwen3.5-4B and Gemma-4-E4B in a statistical tie (macro-F1 difference +0.25 points, 95% CI -0.55 to +1.15).
A single training run can move by about that much from its random seed alone. This notebook repeats the **identical** pilot with training
seed 1 (everything else unchanged: same data split, same LoRA settings, same 3 epochs) to see whether the tie is stable.

**Decision rule, fixed before running** (`src/eval/seed_summary.py`): the tie is *confirmed* if, in both seeds, the macro-F1 difference is under 1 point or its
confidence interval includes 0. If one model is clearly ahead in both seeds, the model choice is reopened. If the seeds disagree, it is reported as mixed.

**Kaggle setup (same as Phase 1):**
1. Dataset from the newest `kaggle_upload/banking-finetuning-code.zip` (it now contains `configs/pilot/*_seed1.yaml` and `src/eval/seed_summary.py`), added to this notebook.
2. Settings: **Accelerator = GPU T4 x2**, **Internet = On**.
3. Run cells top to bottom (or Save Version > Save & Run All). Both models train at the same time, one per GPU. **About 65 to 75 minutes.**

No smoke test and no zero-shot run here: the training and evaluation code and settings are exactly the ones already proven in Phase 1.

In [ ]:
import os, glob, shutil, subprocess, sys
# Kaggle nests uploads differently across versions (and may leave the zip unextracted), so search recursively.
def find_src():
    hits = glob.glob('/kaggle/input/**/src/schema.py', recursive=True)
    if hits:
        return os.path.dirname(os.path.dirname(hits[0]))
    for z in glob.glob('/kaggle/input/**/*.zip', recursive=True):
        shutil.unpack_archive(z, '/kaggle/working/_code')
        hits = glob.glob('/kaggle/working/_code/**/src/schema.py', recursive=True)
        if hits:
            return os.path.dirname(os.path.dirname(hits[0]))
    print('Contents of /kaggle/input:'); print(subprocess.run('find /kaggle/input -maxdepth 4 | head -40', shell=True, capture_output=True, text=True).stdout)
    raise FileNotFoundError('code dataset not found: add banking-finetuning-code via Add Input')

SRC_ROOT = find_src()
PROJ = '/kaggle/working/Banking_FInetuning'
# Always refresh code/configs from the dataset (keeps data/ and outputs/ already in PROJ).
shutil.copytree(SRC_ROOT, PROJ, ignore=shutil.ignore_patterns('__pycache__'), dirs_exist_ok=True)
os.chdir(PROJ)
os.makedirs('logs', exist_ok=True)
os.environ['HF_HOME'] = '/kaggle/tmp/hf'      # model downloads: keep out of /kaggle/working (20 GB cap)
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ['PYTORCH_ALLOC_CONF'] = 'expandable_segments:True'   # less fragmentation / fewer OOMs
print('project at', PROJ, '| code from', SRC_ROOT)
!nvidia-smi --query-gpu=index,name,memory.total,compute_cap --format=csv

In [ ]:
# Freeze Kaggle's CUDA build of torch so pip can't silently replace it with something else.
!pip list --format=freeze | grep -iE '^(torch|torchvision|torchaudio)==' > /tmp/constraints.txt
!cat /tmp/constraints.txt
!pip install -q -U -c /tmp/constraints.txt transformers peft accelerate bitsandbytes mlflow scikit-learn pyyaml pandas datasets

In [ ]:
# GPU check — must pass before anything else. If it fails, read the printed lines.
import subprocess, torch, transformers, peft, bitsandbytes
print('torch', torch.__version__, '| built for CUDA:', torch.version.cuda, '| cuda available:', torch.cuda.is_available(),
      '| GPUs:', torch.cuda.device_count())
print('transformers', transformers.__version__, '| peft', peft.__version__, '| bnb', bitsandbytes.__version__)
print(subprocess.run('nvidia-smi -L; pip list 2>/dev/null | grep -iE "^(torch|nvidia-cuda-runtime)"', shell=True, capture_output=True, text=True).stdout)
assert torch.version.cuda is not None, 'CPU-only torch is installed (see pip list above)'
assert torch.cuda.is_available(), 'No GPU visible: set Accelerator to GPU T4 x2 in the notebook settings, then restart the session'

### Optional: fast kernels for Qwen3.5
Qwen3.5 uses a "gated delta" attention layer. Without the `flash-linear-attention` package it falls back to a very
slow pure-PyTorch version (the smoke-test log warns about this). Installing it can cut Qwen's time a lot.
If this cell errors, just skip it: results are identical, only slower. (`causal_conv1d` needs compiling; skip it.)

In [ ]:
!pip install -q -c /tmp/constraints.txt flash-linear-attention

In [ ]:
# Helpers
import json, threading, time
MODELS = ['qwen3_5_4b_seed1', 'gemma4_e4b_seed1']
cfg = lambda m: f'configs/pilot/{m}.yaml'

def run(module, *args, gpu=None, log=None):
    cmd = [sys.executable, '-m', module, *map(str, args)]
    env = dict(os.environ)
    if gpu is not None:
        env['CUDA_VISIBLE_DEVICES'] = str(gpu)
    if log is None:   # stream into the notebook
        print('$', ' '.join(cmd))
        p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env=env)
        for line in p.stdout:
            print(line, end='')
        rc = p.wait()
    else:
        log.write('$ ' + ' '.join(cmd) + '\n'); log.flush()
        rc = subprocess.call(cmd, stdout=log, stderr=subprocess.STDOUT, env=env)
    return rc

def run_models_parallel(steps_for, tag, models=None):
    """steps_for(model) -> list of (module, args). One model per GPU if 2 GPUs, else one after another."""
    models = models or MODELS
    ngpu = torch.cuda.device_count()
    results = {}
    def worker(m, gpu):
        with open(f'logs/{tag}_{m}.log', 'w') as log:
            for module, args in steps_for(m):
                rc = run(module, *args, gpu=gpu, log=log)
                if rc:
                    results[m] = f'FAILED in {module} (exit {rc})'; return
        results[m] = 'ok'
    def clean(m):   # log lines without tqdm progress-bar noise
        f = f'logs/{tag}_{m}.log'
        return [l for l in open(f).read().splitlines() if 'it/s' not in l and 's/it' not in l] if os.path.exists(f) else []
    def show():
        for m in models:
            print(f'[{m}]', ' | '.join(l[:150] for l in clean(m)[-2:]))
    if ngpu >= 2:
        ts = [threading.Thread(target=worker, args=(m, i)) for i, m in enumerate(models)]
        [t.start() for t in ts]
        while any(t.is_alive() for t in ts):
            time.sleep(60); show(); print('---')
        [t.join() for t in ts]
    else:
        for m in models:
            worker(m, 0); show()
    print(results)
    for m, r in results.items():
        if r != 'ok':
            print(f'\n===== {m}: {r} - last log lines =====')
            print('\n'.join(clean(m)[-45:]))
    return results

## 1. Data (must be identical to the first run)
The split is deterministic (seed 42), so it should be the same. The check below compares fingerprints with the ones from the first run.

In [ ]:
import hashlib
if not os.path.exists('data/stats.json'):
    assert run('src.data.prepare') == 0
EXPECTED = {'data/pilot_train.jsonl': 'e296ea0b27b3ab240ed64c8e390b1249',
            'data/test.jsonl': 'ef8246db4071590d9ac4a876fd494889',
            'data/val.jsonl': 'ea4e8b1aa1d54cf1bc4bff8211dd5e64'}
for f, want in EXPECTED.items():
    got = hashlib.md5(open(f, 'rb').read().replace(b'\r\n', b'\n')).hexdigest()
    print(f.ljust(26), 'OK' if got == want else 'DIFFERENT (stop and tell Claude)', got)
    assert got == want, f'{f} differs from the data used in the first pilot run'

## 2. Train and test both models (seed 1)
Per model: QLoRA on the pilot set, then the full 3,080-example test set. Progress is printed every minute.

In [ ]:
def pilot_steps(m):
    return [('src.train.qlora', ['--config', cfg(m)]),
            ('src.eval.hf_eval', ['--config', cfg(m), '--split', 'data/test.jsonl', '--batch-size', 16,
                                  '--adapter', f'outputs/pilot/{m}/adapter', '--tag', 'finetuned_test'])]
pilot = run_models_parallel(pilot_steps, 'pilot_seed1')

## 3. Save right away
So a dropped session cannot lose the results. The link only works if it is relative to `/kaggle/working`, so the cell switches there first.
If the link still says "not found", use the **Output** panel on the right (Data > Output > `/kaggle/working`, three dots next to the zip > Download),
or the optional Hugging Face upload below. After *Save Version > Save & Run All*, the zip is under the finished version's **Output** tab.

In [ ]:
from IPython.display import FileLink, display
here = os.getcwd()
os.chdir('/kaggle/working')
r = subprocess.run('rm -f pilot_seed1_results.zip && zip -qr pilot_seed1_results.zip Banking_FInetuning/outputs/pilot Banking_FInetuning/logs Banking_FInetuning/data/stats.json -x "*/trainer/*" && ls -lh /kaggle/working/*.zip',
                   shell=True, capture_output=True, text=True)
print(r.stdout, r.stderr)
assert os.path.exists('/kaggle/working/pilot_seed1_results.zip'), 'zip was not created: copy the message above to Claude'
display(FileLink('pilot_seed1_results.zip'))
os.chdir(here)

In [ ]:
# Optional: private upload to YOUR Hugging Face account (survives a dead session). Needs a Kaggle Secret HF_TOKEN with write access; skips if absent.
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret('HF_TOKEN')
except Exception:
    token = None
    print('No HF_TOKEN secret: skipping the Hugging Face upload.')
if token:
    from huggingface_hub import HfApi
    api = HfApi(token=token)
    repo_id = f"{api.whoami()['name']}/banking-triage-pilot-seed1"
    api.create_repo(repo_id, private=True, exist_ok=True)
    api.upload_file(path_or_fileobj='/kaggle/working/pilot_seed1_results.zip', path_in_repo='pilot_seed1_results.zip', repo_id=repo_id)
    print('uploaded (private) ->', f'https://huggingface.co/{repo_id}')

## 4. Seed-1 comparison
Same table as the first pilot, for seed 1 only. (The combined two-seed verdict is computed on your PC, where the seed-0 results are:
`python -m src.eval.seed_summary --models qwen3_5_4b gemma4_e4b --roots pilot_results/outputs/pilot <seed1 outputs/pilot> --suffixes "" _seed1`.)

In [ ]:
assert run('src.eval.compare', '--models', *MODELS, '--out', 'results/pilot/comparison_seed1.md') == 0

In [ ]:
for m in MODELS:
    print(m, json.load(open(f'outputs/pilot/{m}/finetuned_test.json')))